In [1]:
import os
import numpy as np
import pandas as pd

from scipy.stats import skew
from scipy.stats import kurtosis

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
dataset_path = "/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/raw/UAH-DRIVESET-v1"

print(dataset_path)

/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/raw/UAH-DRIVESET-v1


In [3]:
def load_accelerometer(acc_path):

    acc_columns = [

        "timestamp",
        "active",

        "acc_x",
        "acc_y",
        "acc_z",

        "acc_x_kf",
        "acc_y_kf",
        "acc_z_kf",

        "roll",
        "pitch",
        "yaw"

    ]

    acc_df = pd.read_csv(
        acc_path,
        sep=r"\s+",
        header=None,
        names=acc_columns
    )

    return acc_df

def load_gps(gps_path):

    gps_columns = [

        "timestamp",
        "speed",
        "latitude",
        "longitude",
        "altitude",

        "gps_quality",
        "satellites",

        "heading",

        "extra_1",
        "extra_2",
        "extra_3",
        "extra_4"

    ]

    gps_df = pd.read_csv(
        gps_path,
        sep=r"\s+",
        header=None,
        names=gps_columns
    )

    return gps_df
def load_lane_detection(lane_path):

    lane_df = pd.read_csv(
        lane_path,
        sep=r"\s+",
        header=None
    )

    lane_df.columns = [

        "time",

        "lane_offset",

        "phi",

        "road_width",

        "lane_state"
    ]

    return lane_df
def load_vehicle_detection(vehicle_path):

    vehicle_df = pd.read_csv(

        vehicle_path,

        sep=r"\s+",

        header=None

    )

    vehicle_df.columns = [

        "time",

        "front_distance",

        "relative_speed",

        "vehicle_state",

        "confidence"

    ]

    return vehicle_df

In [4]:
def clean_lane_detection(lane_df):

    lane_df = lane_df.copy()

    lane_df.replace(-9, np.nan, inplace=True)

    lane_df.replace(-99, np.nan, inplace=True)

    lane_df = lane_df.sort_values(
        "time"
    )

    lane_df = lane_df.reset_index(
        drop=True
    )

    return lane_df

def load_lane_detection(lane_path):

    lane_df = pd.read_csv(
        lane_path,
        sep=r"\s+",
        header=None
    )

    lane_df.columns = [

        "time",

        "lane_offset",

        "phi",

        "road_width",

        "lane_state"
    ]

    return lane_df


In [5]:
def load_vehicle_detection(vehicle_path):

    vehicle_df = pd.read_csv(

        vehicle_path,

        sep=r"\s+",

        header=None

    )

    vehicle_df.columns = [

        "time",

        "front_distance",

        "relative_speed",

        "vehicle_state",

        "confidence"

    ]

    return vehicle_df

def clean_vehicle_detection(vehicle_df):

    vehicle_df = vehicle_df.copy()

    vehicle_df = vehicle_df.sort_values(

        "time"

    )

    vehicle_df = vehicle_df.reset_index(

        drop=True

    )

    return vehicle_df

In [6]:
def synchronize_sensors(acc_df, gps_df):

    master_df = pd.merge_asof(

        acc_df.sort_values("timestamp"),

        gps_df.sort_values("timestamp"),

        on="timestamp",

        direction="nearest"

    )

    return master_df


In [7]:
def merge_lane_data(master_df, lane_df):

    master_df = master_df.copy()
    lane_df = lane_df.copy()

    master_df = master_df.sort_values(
        "timestamp"
    )

    lane_df = lane_df.sort_values(
        "time"
    )

    merged_df = pd.merge_asof(

        master_df,

        lane_df,

        left_on="timestamp",

        right_on="time",

        direction="nearest"

    )

    merged_df.drop(
        columns=["time"],
        inplace=True
    )

    return merged_df

In [8]:
def merge_vehicle_data(master_df, vehicle_df):

    master_df = master_df.copy()
    vehicle_df = vehicle_df.copy()

    master_df = master_df.sort_values(
        "timestamp"
    )

    vehicle_df = vehicle_df.sort_values(
        "time"
    )

    merged_df = pd.merge_asof(

        master_df,

        vehicle_df,

        left_on="timestamp",

        right_on="time",

        direction="nearest"

    )

    merged_df.drop(
        columns=["time"],
        inplace=True
    )

    return merged_df

In [9]:
sample_driver = "D1"

sample_trip = "20151110175712-16km-D1-NORMAL1-SECONDARY"

trip_path = os.path.join(
    dataset_path,
    sample_driver,
    sample_trip
)

acc_path = os.path.join(
    trip_path,
    "RAW_ACCELEROMETERS.txt"
)

gps_path = os.path.join(
    trip_path,
    "RAW_GPS.txt"
)

lane_path = os.path.join(
    trip_path,
    "PROC_LANE_DETECTION.txt"
)

vehicle_path = os.path.join(
    trip_path,
    "PROC_VEHICLE_DETECTION.txt"
)

acc_df = load_accelerometer(acc_path)

gps_df = load_gps(gps_path)

lane_df = clean_lane_detection(
    load_lane_detection(lane_path)
)

vehicle_df = clean_vehicle_detection(
    load_vehicle_detection(vehicle_path)
)

master_df = synchronize_sensors(
    acc_df,
    gps_df
)

master_df = merge_lane_data(
    master_df,
    lane_df
)

master_df = merge_vehicle_data(
    master_df,
    vehicle_df
)

print(master_df.shape)

master_df.head()

(6170, 30)


,timestamp,active,acc_x,acc_y,acc_z,acc_x_kf,acc_y_kf,acc_z_kf,roll,pitch,...,extra_3,extra_4,lane_offset,phi,road_width,lane_state,front_distance,relative_speed,vehicle_state,confidence
0,6.94,1,0.017,-0.011,0.018,-0.005,0.008,0.018,-1.523,0.015,...,0,0,0.0,-0.0,3.5,-1.0,-1.00,-1.00,0,67.4
1,7.03,1,0.046,0.007,0.019,0.016,-0.002,0.018,-1.522,0.012,...,0,0,0.0,-0.0,3.5,-1.0,-1.00,-1.00,0,67.4
2,7.14,1,0.052,-0.016,0.027,0.037,-0.005,0.018,-1.520,0.014,...,0,0,0.0,-0.0,3.5,-1.0,-1.00,-1.00,0,67.4
3,7.24,1,0.015,-0.016,0.026,0.038,-0.009,0.024,-1.523,0.014,...,0,0,0.0,-0.0,3.5,-1.0,17.97,0.99,2,65.2
4,7.34,1,-0.014,-0.017,0.040,0.012,-0.016,0.032,-1.525,0.012,...,0,0,0.0,-0.0,3.5,-1.0,17.97,0.99,2,65.2


In [10]:
print(master_df.columns)

print()

master_df[
    [
        "timestamp",
        "lane_offset",
        "phi",
        "front_distance",
        "relative_speed",
        "vehicle_state",
        "confidence"
    ]
].head(20)

Index(['timestamp', 'active', 'acc_x', 'acc_y', 'acc_z', 'acc_x_kf',
       'acc_y_kf', 'acc_z_kf', 'roll', 'pitch', 'yaw', 'speed', 'latitude',
       'longitude', 'altitude', 'gps_quality', 'satellites', 'heading',
       'extra_1', 'extra_2', 'extra_3', 'extra_4', 'lane_offset', 'phi',
       'road_width', 'lane_state', 'front_distance', 'relative_speed',
       'vehicle_state', 'confidence'],
      dtype='object')



,timestamp,lane_offset,phi,front_distance,relative_speed,vehicle_state,confidence
0,6.94,0.0,-0.0,-1.00,-1.00,0,67.4
1,7.03,0.0,-0.0,-1.00,-1.00,0,67.4
2,7.14,0.0,-0.0,-1.00,-1.00,0,67.4
3,7.24,0.0,-0.0,17.97,0.99,2,65.2
4,7.34,0.0,-0.0,17.97,0.99,2,65.2
5,7.44,0.0,-0.0,18.04,1.00,2,65.2
6,7.54,0.0,-0.0,18.12,1.00,2,65.2
7,7.64,0.0,-0.0,18.10,1.00,2,65.2
8,7.74,0.0,-0.0,18.07,1.00,2,65.2
9,7.85,0.0,-0.0,18.16,1.00,2,65.2


In [13]:
def engineer_features_v3(master_df):

    master_df = master_df.copy()

    # ----------------------------------------------------
    # Acceleration Features
    # ----------------------------------------------------

    master_df["acc_resultant"] = np.sqrt(
        master_df["acc_x"]**2 +
        master_df["acc_y"]**2 +
        master_df["acc_z"]**2
    )

    master_df["acc_horizontal"] = np.sqrt(
        master_df["acc_x"]**2 +
        master_df["acc_y"]**2
    )

    master_df["acc_vertical"] = master_df["acc_z"]

    # ----------------------------------------------------
    # Delta Features
    # ----------------------------------------------------

    master_df["speed_delta"] = master_df["speed"].diff().fillna(0)

    master_df["heading_delta"] = master_df["heading"].diff().fillna(0)

    master_df["roll_delta"] = master_df["roll"].diff().fillna(0)

    master_df["pitch_delta"] = master_df["pitch"].diff().fillna(0)

    master_df["yaw_delta"] = master_df["yaw"].diff().fillna(0)

    # ----------------------------------------------------
    # Lane Features
    # ----------------------------------------------------

    master_df["lane_departure"] = (
        master_df["lane_offset"].abs() > 0.75
    ).astype(int)

    master_df["steering_alignment"] = (
        master_df["lane_offset"] *
        master_df["phi"]
    )

    return master_df

In [14]:
feature_df = engineer_features_v3(master_df)

feature_df[
    [
        "lane_offset",
        "phi",
        "lane_departure",
        "steering_alignment"
    ]
].head(20)

,lane_offset,phi,lane_departure,steering_alignment
0,0.0,-0.0,0,-0.0
1,0.0,-0.0,0,-0.0
2,0.0,-0.0,0,-0.0
3,0.0,-0.0,0,-0.0
4,0.0,-0.0,0,-0.0
5,0.0,-0.0,0,-0.0
6,0.0,-0.0,0,-0.0
7,0.0,-0.0,0,-0.0
8,0.0,-0.0,0,-0.0
9,0.0,-0.0,0,-0.0


In [15]:
def engineer_features_v3(master_df):

    master_df = master_df.copy()

    # ----------------------------------------------------
    # Acceleration Features
    # ----------------------------------------------------

    master_df["acc_resultant"] = np.sqrt(
        master_df["acc_x"]**2 +
        master_df["acc_y"]**2 +
        master_df["acc_z"]**2
    )

    master_df["acc_horizontal"] = np.sqrt(
        master_df["acc_x"]**2 +
        master_df["acc_y"]**2
    )

    master_df["acc_vertical"] = master_df["acc_z"]

    # ----------------------------------------------------
    # Delta Features
    # ----------------------------------------------------

    master_df["speed_delta"] = master_df["speed"].diff().fillna(0)

    master_df["heading_delta"] = master_df["heading"].diff().fillna(0)

    master_df["roll_delta"] = master_df["roll"].diff().fillna(0)

    master_df["pitch_delta"] = master_df["pitch"].diff().fillna(0)

    master_df["yaw_delta"] = master_df["yaw"].diff().fillna(0)

    # ----------------------------------------------------
    # Lane Features
    # ----------------------------------------------------

    master_df["lane_departure"] = (
        master_df["lane_offset"].abs() > 0.75
    ).astype(int)

    master_df["steering_alignment"] = (
        master_df["lane_offset"] *
        master_df["phi"]
    )
    # ----------------------------------------------------
    # Vehicle Features
    # ----------------------------------------------------

    # Önde araç var mı?
    master_df["vehicle_present"] = (
        master_df["front_distance"] > 0
    ).astype(int)

    # Geçerli takip mesafesi
    master_df["safe_distance"] = np.where(
        master_df["front_distance"] > 0,
        master_df["front_distance"],
        np.nan
    )

    # Yaklaşma hızı
    master_df["closing_speed"] = np.where(
        master_df["relative_speed"] > 0,
        master_df["relative_speed"],
        0
    )

    # Time To Collision (TTC)
    master_df["ttc"] = np.where(
        (master_df["front_distance"] > 0) &
        (master_df["relative_speed"] > 0.1),
        master_df["front_distance"] /
        master_df["relative_speed"],
        np.nan
    )

    # Confidence'ı normalize et
    master_df["confidence"] = (
        master_df["confidence"] / 100
    )
    return master_df

In [16]:
feature_df = engineer_features_v3(master_df)

feature_df[
    [
        "front_distance",
        "relative_speed",
        "vehicle_present",
        "safe_distance",
        "closing_speed",
        "ttc",
        "confidence"
    ]
].head(20)

,front_distance,relative_speed,vehicle_present,safe_distance,closing_speed,ttc,confidence
0,-1.00,-1.00,0,NaN,0.00,NaN,0.674
1,-1.00,-1.00,0,NaN,0.00,NaN,0.674
2,-1.00,-1.00,0,NaN,0.00,NaN,0.674
3,17.97,0.99,1,17.97,0.99,18.151515,0.652
4,17.97,0.99,1,17.97,0.99,18.151515,0.652
5,18.04,1.00,1,18.04,1.00,18.040000,0.652
6,18.12,1.00,1,18.12,1.00,18.120000,0.652
7,18.10,1.00,1,18.10,1.00,18.100000,0.652
8,18.07,1.00,1,18.07,1.00,18.070000,0.652
9,18.16,1.00,1,18.16,1.00,18.160000,0.652


In [17]:
window_features_v4 = [

    # Acceleration
    "acc_resultant",
    "acc_horizontal",

    # GPS
    "speed",
    "speed_delta",

    "roll",
    "pitch",
    "yaw",

    # Lane
    "lane_offset",
    "phi",
    "steering_alignment",

    # Vehicle
    "front_distance",
    "relative_speed",
    "closing_speed",
    "ttc",

    # Binary Features
    "vehicle_present",
    "lane_departure"
]

print(window_features_v4)

['acc_resultant', 'acc_horizontal', 'speed', 'speed_delta', 'roll', 'pitch', 'yaw', 'lane_offset', 'phi', 'steering_alignment', 'front_distance', 'relative_speed', 'closing_speed', 'ttc', 'vehicle_present', 'lane_departure']


In [18]:
def extract_window_features_v4(window, feature_list):

    window_stats = {}

    for feature in feature_list:

        stats = extract_statistics_v4(
            window[feature],
            feature
        )

        for stat_name, stat_value in stats.items():

            column_name = f"{feature}_{stat_name}"

            window_stats[column_name] = stat_value

    return window_stats

In [20]:
def extract_statistics_v4(signal, feature_name):

    features = {}

    binary_features = [

        "vehicle_present",

        "lane_departure"

    ]

    if feature_name in binary_features:

        features["mean"] = signal.mean()

        return features

    # ----------------------------------------------------
    # Continuous Features
    # ----------------------------------------------------

    features["mean"] = signal.mean()
    features["std"] = signal.std()
    features["variance"] = signal.var()

    features["min"] = signal.min()
    features["max"] = signal.max()

    features["median"] = signal.median()

    features["rms"] = np.sqrt(
        np.mean(signal ** 2)
    )

    features["skewness"] = signal.skew()

    features["kurtosis"] = signal.kurt()

    features["q25"] = signal.quantile(0.25)

    features["q75"] = signal.quantile(0.75)

    features["iqr"] = (
        features["q75"] -
        features["q25"]
    )

    return features

In [21]:
def create_sliding_windows_v4(
    feature_df,
    feature_list,
    window_size
):

    all_window_features = []

    for start in range(
        0,
        len(feature_df) - window_size + 1
    ):

        end = start + window_size

        window = feature_df.iloc[start:end]

        window_stats = extract_window_features_v4(
            window,
            feature_list
        )

        all_window_features.append(
            window_stats
        )

    return pd.DataFrame(
        all_window_features
    )

In [23]:
feature_df = engineer_features_v3(master_df)

print(feature_df.shape)

feature_df.head()

(6170, 44)


,timestamp,active,acc_x,acc_y,acc_z,acc_x_kf,acc_y_kf,acc_z_kf,roll,pitch,...,heading_delta,roll_delta,pitch_delta,yaw_delta,lane_departure,steering_alignment,vehicle_present,safe_distance,closing_speed,ttc
0,6.94,1,0.017,-0.011,0.018,-0.005,0.008,0.018,-1.523,0.015,...,0.0,0.000,0.000,0.000,0,-0.0,0,NaN,0.00,NaN
1,7.03,1,0.046,0.007,0.019,0.016,-0.002,0.018,-1.522,0.012,...,0.0,0.001,-0.003,0.000,0,-0.0,0,NaN,0.00,NaN
2,7.14,1,0.052,-0.016,0.027,0.037,-0.005,0.018,-1.520,0.014,...,0.0,0.002,0.002,-0.001,0,-0.0,0,NaN,0.00,NaN
3,7.24,1,0.015,-0.016,0.026,0.038,-0.009,0.024,-1.523,0.014,...,0.0,-0.003,0.000,0.000,0,-0.0,1,17.97,0.99,18.151515
4,7.34,1,-0.014,-0.017,0.040,0.012,-0.016,0.032,-1.525,0.012,...,0.0,-0.002,-0.002,0.000,0,-0.0,1,17.97,0.99,18.151515


In [24]:
window_dataset = create_sliding_windows_v4(
    feature_df,
    window_features_v4,
    60
)

print(window_dataset.shape)

window_dataset.head()

(6111, 170)


,acc_resultant_mean,acc_resultant_std,acc_resultant_variance,acc_resultant_min,acc_resultant_max,acc_resultant_median,acc_resultant_rms,acc_resultant_skewness,acc_resultant_kurtosis,acc_resultant_q25,...,ttc_max,ttc_median,ttc_rms,ttc_skewness,ttc_kurtosis,ttc_q25,ttc_q75,ttc_iqr,vehicle_present_mean,lane_departure_mean
0,0.052610,0.022257,0.000495,0.019339,0.118106,0.048171,0.057052,0.794801,0.456685,0.034369,...,18.16,17.565789,17.497171,0.046556,-1.628589,16.988947,17.920792,0.931845,0.916667,0.0
1,0.052771,0.022104,0.000489,0.019339,0.118106,0.048171,0.057142,0.809346,0.504046,0.035270,...,18.16,17.449561,17.490740,0.080675,-1.622980,16.989039,17.920792,0.931753,0.933333,0.0
2,0.052418,0.022313,0.000498,0.019339,0.118106,0.046448,0.056897,0.814978,0.446186,0.034369,...,18.16,17.333333,17.483376,0.115324,-1.620604,16.989130,17.920792,0.931662,0.950000,0.0
3,0.052048,0.022357,0.000500,0.019339,0.118106,0.045901,0.056573,0.856475,0.479302,0.034369,...,18.16,17.331140,17.477934,0.145848,-1.607655,16.999912,17.920392,0.920480,0.966667,0.0
4,0.052207,0.022260,0.000496,0.019339,0.118106,0.045901,0.056681,0.853881,0.508790,0.035270,...,18.16,17.326974,17.460581,0.202135,-1.578579,16.999912,17.912166,0.912254,0.966667,0.0


In [25]:
window_dataset.columns[-40:]

Index(['front_distance_q75', 'front_distance_iqr', 'relative_speed_mean',
       'relative_speed_std', 'relative_speed_variance', 'relative_speed_min',
       'relative_speed_max', 'relative_speed_median', 'relative_speed_rms',
       'relative_speed_skewness', 'relative_speed_kurtosis',
       'relative_speed_q25', 'relative_speed_q75', 'relative_speed_iqr',
       'closing_speed_mean', 'closing_speed_std', 'closing_speed_variance',
       'closing_speed_min', 'closing_speed_max', 'closing_speed_median',
       'closing_speed_rms', 'closing_speed_skewness', 'closing_speed_kurtosis',
       'closing_speed_q25', 'closing_speed_q75', 'closing_speed_iqr',
       'ttc_mean', 'ttc_std', 'ttc_variance', 'ttc_min', 'ttc_max',
       'ttc_median', 'ttc_rms', 'ttc_skewness', 'ttc_kurtosis', 'ttc_q25',
       'ttc_q75', 'ttc_iqr', 'vehicle_present_mean', 'lane_departure_mean'],
      dtype='object')

In [30]:
trip_list = []

for driver in sorted(os.listdir(dataset_path)):

    if not driver.startswith("D"):
        continue

    driver_path = os.path.join(dataset_path, driver)

    if not os.path.isdir(driver_path):
        continue

    for trip in sorted(os.listdir(driver_path)):

        trip_path = os.path.join(driver_path, trip)

        if os.path.isdir(trip_path):

            trip_list.append({
                "driver": driver,
                "trip": trip
            })

print(len(trip_list))

40


In [31]:
from sklearn.model_selection import train_test_split

train_trips, test_trips = train_test_split(
    trip_list,
    test_size=0.20,
    random_state=42
)

print(len(train_trips))
print(len(test_trips))

32
8


In [26]:
def process_trip_v5(
    dataset_path,
    driver,
    trip,
    window_size=60
):

    trip_path = os.path.join(
        dataset_path,
        driver,
        trip
    )

    # -----------------------------
    # File Paths
    # -----------------------------

    acc_path = os.path.join(
        trip_path,
        "RAW_ACCELEROMETERS.txt"
    )

    gps_path = os.path.join(
        trip_path,
        "RAW_GPS.txt"
    )

    lane_path = os.path.join(
        trip_path,
        "PROC_LANE_DETECTION.txt"
    )

    vehicle_path = os.path.join(
        trip_path,
        "PROC_VEHICLE_DETECTION.txt"
    )

    # -----------------------------
    # Load
    # -----------------------------

    acc_df = load_accelerometer(acc_path)

    gps_df = load_gps(gps_path)

    lane_df = clean_lane_detection(
        load_lane_detection(lane_path)
    )

    vehicle_df = clean_vehicle_detection(
        load_vehicle_detection(vehicle_path)
    )

    # -----------------------------
    # Merge Sensors
    # -----------------------------

    master_df = synchronize_sensors(
        acc_df,
        gps_df
    )

    master_df = merge_lane_data(
        master_df,
        lane_df
    )

    master_df = merge_vehicle_data(
        master_df,
        vehicle_df
    )

    # -----------------------------
    # Feature Engineering
    # -----------------------------

    feature_df = engineer_features_v3(
        master_df
    )

    # -----------------------------
    # Sliding Windows
    # -----------------------------

    window_dataset = create_sliding_windows_v4(
        feature_df,
        window_features_v4,
        window_size
    )

    # -----------------------------
    # Metadata
    # -----------------------------

    window_dataset["driver"] = driver

    window_dataset["trip"] = trip

    road_type = trip.split("-")[-1]

    behavior = trip.split("-")[-2]

    window_dataset["road_type"] = road_type

    window_dataset["behavior"] = behavior

    return window_dataset

In [27]:
sample_dataset = process_trip_v5(
    dataset_path,
    "D1",
    "20151110175712-16km-D1-NORMAL1-SECONDARY"
)

print(sample_dataset.shape)

sample_dataset.head()

(6111, 174)


,acc_resultant_mean,acc_resultant_std,acc_resultant_variance,acc_resultant_min,acc_resultant_max,acc_resultant_median,acc_resultant_rms,acc_resultant_skewness,acc_resultant_kurtosis,acc_resultant_q25,...,ttc_kurtosis,ttc_q25,ttc_q75,ttc_iqr,vehicle_present_mean,lane_departure_mean,driver,trip,road_type,behavior
0,0.052610,0.022257,0.000495,0.019339,0.118106,0.048171,0.057052,0.794801,0.456685,0.034369,...,-1.628589,16.988947,17.920792,0.931845,0.916667,0.0,D1,20151110175712-16km-D1-NORMAL1-SECONDARY,SECONDARY,NORMAL1
1,0.052771,0.022104,0.000489,0.019339,0.118106,0.048171,0.057142,0.809346,0.504046,0.035270,...,-1.622980,16.989039,17.920792,0.931753,0.933333,0.0,D1,20151110175712-16km-D1-NORMAL1-SECONDARY,SECONDARY,NORMAL1
2,0.052418,0.022313,0.000498,0.019339,0.118106,0.046448,0.056897,0.814978,0.446186,0.034369,...,-1.620604,16.989130,17.920792,0.931662,0.950000,0.0,D1,20151110175712-16km-D1-NORMAL1-SECONDARY,SECONDARY,NORMAL1
3,0.052048,0.022357,0.000500,0.019339,0.118106,0.045901,0.056573,0.856475,0.479302,0.034369,...,-1.607655,16.999912,17.920392,0.920480,0.966667,0.0,D1,20151110175712-16km-D1-NORMAL1-SECONDARY,SECONDARY,NORMAL1
4,0.052207,0.022260,0.000496,0.019339,0.118106,0.045901,0.056681,0.853881,0.508790,0.035270,...,-1.578579,16.999912,17.912166,0.912254,0.966667,0.0,D1,20151110175712-16km-D1-NORMAL1-SECONDARY,SECONDARY,NORMAL1


In [28]:
sample_dataset.columns[-10:]

Index(['ttc_kurtosis', 'ttc_q25', 'ttc_q75', 'ttc_iqr', 'vehicle_present_mean',
       'lane_departure_mean', 'driver', 'trip', 'road_type', 'behavior'],
      dtype='object')

In [32]:
train_datasets = []

for trip in train_trips:

    print(
        f"Processing Train : {trip['driver']} - {trip['trip']}"
    )

    trip_dataset = process_trip_v5(
        dataset_path,
        trip["driver"],
        trip["trip"],
        60
    )

    train_datasets.append(
        trip_dataset
    )

Processing Train : D6 - 20151221120051-26km-D6-AGGRESSIVE-MOTORWAY
Processing Train : D1 - 20151111135612-13km-D1-DROWSY-SECONDARY
Processing Train : D4 - 20151204152848-25km-D4-NORMAL-MOTORWAY
Processing Train : D2 - 20151120135152-25km-D2-DROWSY-MOTORWAY
Processing Train : D2 - 20151120164606-16km-D2-DROWSY-SECONDARY
Processing Train : D5 - 20151211162829-16km-D5-NORMAL1-SECONDARY
Processing Train : D5 - 20151211170502-16km-D5-DROWSY-SECONDARY
Processing Train : D2 - 20151120133502-26km-D2-AGGRESSIVE-MOTORWAY
Processing Train : D3 - 20151126125458-16km-D3-NORMAL2-SECONDARY
Processing Train : D4 - 20151203175637-17km-D4-DROWSY-SECONDARY
Processing Train : D1 - 20151110175712-16km-D1-NORMAL1-SECONDARY
Processing Train : D5 - 20151211165606-12km-D5-AGGRESSIVE-SECONDARY
Processing Train : D1 - 20151111134545-16km-D1-AGGRESSIVE-SECONDARY
Processing Train : D2 - 20151120162105-17km-D2-NORMAL2-SECONDARY
Processing Train : D1 - 20151110180824-16km-D1-NORMAL2-SECONDARY
Processing Train : D5 -

In [33]:
train_dataset = pd.concat(
    train_datasets,
    ignore_index=True
)

print(train_dataset.shape)

train_dataset.head()

(242965, 174)


,acc_resultant_mean,acc_resultant_std,acc_resultant_variance,acc_resultant_min,acc_resultant_max,acc_resultant_median,acc_resultant_rms,acc_resultant_skewness,acc_resultant_kurtosis,acc_resultant_q25,...,ttc_kurtosis,ttc_q25,ttc_q75,ttc_iqr,vehicle_present_mean,lane_departure_mean,driver,trip,road_type,behavior
0,0.062950,0.037494,0.001406,0.012961,0.178804,0.058821,0.073110,0.994585,0.711390,0.033116,...,NaN,NaN,NaN,NaN,0.000000,0.283333,D6,20151221120051-26km-D6-AGGRESSIVE-MOTORWAY,MOTORWAY,AGGRESSIVE
1,0.060696,0.034344,0.001179,0.012961,0.144686,0.056502,0.069598,0.821860,0.113281,0.033116,...,NaN,NaN,NaN,NaN,0.000000,0.300000,D6,20151221120051-26km-D6-AGGRESSIVE-MOTORWAY,MOTORWAY,AGGRESSIVE
2,0.060563,0.034495,0.001190,0.012961,0.144686,0.056502,0.069556,0.807773,0.093973,0.033116,...,NaN,NaN,NaN,NaN,0.000000,0.316667,D6,20151221120051-26km-D6-AGGRESSIVE-MOTORWAY,MOTORWAY,AGGRESSIVE
3,0.059966,0.034357,0.001180,0.012961,0.144686,0.053694,0.068969,0.863860,0.203253,0.033116,...,NaN,NaN,NaN,NaN,0.000000,0.316667,D6,20151221120051-26km-D6-AGGRESSIVE-MOTORWAY,MOTORWAY,AGGRESSIVE
4,0.059171,0.033793,0.001142,0.012961,0.144686,0.053694,0.068001,0.933059,0.452223,0.033116,...,NaN,19.59633,19.59633,0.0,0.016667,0.316667,D6,20151221120051-26km-D6-AGGRESSIVE-MOTORWAY,MOTORWAY,AGGRESSIVE


In [34]:
test_datasets = []

for trip in test_trips:

    print(
        f"Processing Test : {trip['driver']} - {trip['trip']}"
    )

    trip_dataset = process_trip_v5(
        dataset_path,
        trip["driver"],
        trip["trip"],
        60
    )

    test_datasets.append(
        trip_dataset
    )

Processing Test : D3 - 20151126132013-17km-D3-DROWSY-SECONDARY
Processing Test : D3 - 20151126124208-16km-D3-NORMAL1-SECONDARY
Processing Test : D3 - 20151126113754-26km-D3-DROWSY-MOTORWAY
Processing Test : D4 - 20151204154908-25km-D4-AGGRESSIVE-MOTORWAY
Processing Test : D1 - 20151111132348-25km-D1-DROWSY-MOTORWAY
Processing Test : D2 - 20151120163350-16km-D2-AGGRESSIVE-SECONDARY
Processing Test : D6 - 20151221112434-17km-D6-NORMAL-SECONDARY
Processing Test : D4 - 20151204160823-25km-D4-DROWSY-MOTORWAY


In [35]:
test_dataset = pd.concat(
    test_datasets,
    ignore_index=True
)

print(test_dataset.shape)

test_dataset.head()

(66060, 174)


,acc_resultant_mean,acc_resultant_std,acc_resultant_variance,acc_resultant_min,acc_resultant_max,acc_resultant_median,acc_resultant_rms,acc_resultant_skewness,acc_resultant_kurtosis,acc_resultant_q25,...,ttc_kurtosis,ttc_q25,ttc_q75,ttc_iqr,vehicle_present_mean,lane_departure_mean,driver,trip,road_type,behavior
0,0.069472,0.041229,0.001700,0.009487,0.146826,0.056582,0.080609,0.716307,-0.647953,0.040467,...,NaN,NaN,NaN,NaN,0.0,0.0,D3,20151126132013-17km-D3-DROWSY-SECONDARY,SECONDARY,DROWSY
1,0.067631,0.040168,0.001613,0.009487,0.146826,0.055759,0.078489,0.781847,-0.482125,0.039345,...,NaN,NaN,NaN,NaN,0.0,0.0,D3,20151126132013-17km-D3-DROWSY-SECONDARY,SECONDARY,DROWSY
2,0.066677,0.040049,0.001604,0.009487,0.146826,0.053646,0.077608,0.849096,-0.373484,0.039345,...,NaN,NaN,NaN,NaN,0.0,0.0,D3,20151126132013-17km-D3-DROWSY-SECONDARY,SECONDARY,DROWSY
3,0.064657,0.038979,0.001519,0.009487,0.146826,0.051527,0.075330,0.909001,-0.183656,0.037702,...,NaN,NaN,NaN,NaN,0.0,0.0,D3,20151126132013-17km-D3-DROWSY-SECONDARY,SECONDARY,DROWSY
4,0.062385,0.038071,0.001449,0.009487,0.146826,0.050660,0.072919,0.934266,-0.004782,0.035844,...,NaN,NaN,NaN,NaN,0.0,0.0,D3,20151126132013-17km-D3-DROWSY-SECONDARY,SECONDARY,DROWSY


In [36]:
print(train_dataset.shape)
print(test_dataset.shape)

print()

print(
    train_dataset.columns.equals(
        test_dataset.columns
    )
)

(242965, 174)
(66060, 174)

True


In [37]:
train_dataset.to_csv(
    "/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/processed/train_dataset_v5_lane_vehicle.csv",
    index=False
)

test_dataset.to_csv(
    "/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/processed/test_dataset_v5_lane_vehicle.csv",
    index=False
)

print("Dataset saved successfully!")

Dataset saved successfully!


In [38]:
print(train_dataset.isnull().sum().sort_values(ascending=False).head(20))

ttc_kurtosis                   102637
ttc_skewness                   100639
ttc_variance                    98965
ttc_std                         98965
ttc_mean                        97360
ttc_min                         97360
ttc_median                      97360
ttc_max                         97360
ttc_rms                         97360
ttc_q25                         97360
ttc_q75                         97360
ttc_iqr                         97360
lane_offset_kurtosis             3093
phi_kurtosis                     3093
steering_alignment_kurtosis      3093
lane_offset_skewness             3058
phi_skewness                     3058
steering_alignment_skewness      3058
steering_alignment_variance      3023
steering_alignment_std           3023
dtype: int64


In [39]:
print(test_dataset.isnull().sum().sort_values(ascending=False).head(20))

ttc_kurtosis                   31906
ttc_skewness                   31345
ttc_variance                   30758
ttc_std                        30758
ttc_mean                       30259
ttc_min                        30259
ttc_median                     30259
ttc_max                        30259
ttc_rms                        30259
ttc_q25                        30259
ttc_q75                        30259
ttc_iqr                        30259
lane_offset_kurtosis             552
phi_kurtosis                     552
steering_alignment_kurtosis      552
lane_offset_skewness             544
phi_skewness                     544
steering_alignment_skewness      544
steering_alignment_variance      536
steering_alignment_std           536
dtype: int64
